In [10]:
import pandas
import numpy as np
import pandas as pd
from huggingface_hub.keras_mixin import keras
from tensorflow.keras.models import Sequential,Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.layers import Conv2D,MaxPooling2D,Flatten,Dense
from tensorflow.keras.utils import to_categorical
import keras
import os
import unicodedata
import re

In [11]:
from datasets import load_dataset

ds = load_dataset("opus100", "en-fa")

In [12]:
def unicode_to_ascii(s):
    return ''.join(c for c in unicodedata.normalize('NFD',s)if unicodedata.category(c) !='MN')

In [13]:
en_list=[]
fa_list=[]
train_data=ds['train']
for   items in  train_data:
    en = items['translation']['en'].lower().strip()
    en= re.sub(r"([.!?~,])", r" \1", en)
    en= re.sub(r'([" "])+', " ", en)
    en= '<start> ' + en + ' <end>'

    fa=items['translation']['fa'].lower().strip()
    fa=re.sub(r"\s+",' ',fa)
    en_list.append(en)
    fa_list.append(fa)

In [14]:
df=pd.DataFrame({'fa':fa_list,'en':en_list})
df. to_csv("fa-en-train.csv",index=False,encoding='utf-8')

In [15]:
from transformers import AutoTokenizer
import tensorflow as tf
batch_size=100
x_batch=[]
y_batch=[]
tokenizer_en=AutoTokenizer.from_pretrained("t5-small", cache_dir="C:/local_model")
tokenizer_fa=AutoTokenizer.from_pretrained("t5-small", cache_dir="C:/local_model")

for i in range(0,len(df),batch_size):
    en_inputs=tokenizer_en(df['en'].iloc[i:i+batch_size].tolist(),
                       padding=True,
                       truncation=True,
                       return_tensors="tf",)

    fa_labels=tokenizer_fa(df['fa'].iloc[i:i+batch_size].tolist(),
                       padding=True,
                       truncation=True,
                       return_tensors="tf")

    x_batch.append(en_inputs['input_ids'])
    y_batch.append(fa_labels['input_ids'])

print(x_batch[0].shape,y_batch[0].shape)

tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

(100, 49) (100, 74)


In [ ]:
embdding_dim=256
units=512
vocab_size_en=tokenizer_en.vocab_size
vocab_size_fa=tokenizer_fa.vocab_size